# Social Profile Gender Classification

This notebook builds a machine-learning pipeline for an **educational binary-classification exercise** using profile text and metadata.

The dataset contains a mixture of:
- profile text (`fullname`, `username`, `biography`)
- numerical features (`follower_count`, `following_count`)
- categorical features (`age`)
- boolean profile flags (`is_business`, `is_verified`, `is_private`)

The main goal is to demonstrate a clean heterogeneous-feature pipeline with `ColumnTransformer`, TF-IDF, and `LinearSVC`.

> **Responsible-use note:** Gender is a sensitive personal attribute. This project is presented as an educational ML exercise only. Predictions may encode cultural, linguistic, and sampling biases and should not be used to make decisions about people.


## 1. Imports


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC


## 2. Load data

Place `train_data.csv` and `test_data.csv` inside the `data/` directory.


In [ ]:
DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


## 3. Quick inspection


In [ ]:
train.head()


In [ ]:
train.info()


In [ ]:
print("Duplicate rows:", train.duplicated().sum())

print("\nMissing values:")
print(train.isna().sum())

print("\nTarget distribution:")
print(train["gender"].value_counts())
print(train["gender"].value_counts(normalize=True).round(3))


The original dataset contains 8,000 training rows. Only `is_business` had a very small number of missing values in the original run. Missing-value handling is kept **inside the pipeline**, so the same preprocessing is applied consistently to training, validation, and test data.


## 4. Train / validation split


In [ ]:
X = train.drop(columns="gender")
y = train["gender"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))


## 5. Feature design

Different feature types carry different kinds of signal:

- **`fullname`** — character patterns are useful because spelling, prefixes, suffixes, and transliterations matter.
- **`username`** — character n-grams work well with shortened names, underscores, digits, and spelling variants.
- **`biography`** — word-level TF-IDF captures meaningful words and short phrases.
- **follower/following counts** — highly skewed, so `log1p` is applied before scaling.
- **`age`** — treated as a categorical variable.
- **profile flags** — imputed and passed through as numeric binary features.


In [ ]:
numeric_cols = ["follower_count", "following_count"]
categorical_cols = ["age"]
boolean_cols = ["is_business", "is_verified", "is_private"]


### Numerical pipeline

Follower counts are strongly right-skewed, so the pipeline applies:

`median imputation → log1p → StandardScaler`


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "log1p",
        FunctionTransformer(
            np.log1p,
            feature_names_out="one-to-one",
        ),
    ),
    ("scaler", StandardScaler()),
])


### Categorical and boolean pipelines


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

boolean_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])


### Text vectorizers

The original experiment found a strong representation using:
- character TF-IDF for `fullname`
- character TF-IDF for `username`
- word unigram + bigram TF-IDF for `biography`


In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    return str(text).lower().strip()


name_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    preprocessor=normalize_text,
)

username_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    preprocessor=normalize_text,
)

bio_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    preprocessor=normalize_text,
)


## 6. Combined preprocessing pipeline


In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
    ("boolean", boolean_pipeline, boolean_cols),
    ("fullname", name_tfidf, "fullname"),
    ("username", username_tfidf, "username"),
    ("biography", bio_tfidf, "biography"),
])


The complete feature flow is:

```text
follower_count ─┐
following_count ─┴─> impute → log1p → scale

age ────────────────> impute → one-hot encode

boolean flags ──────> impute

fullname ───────────> character TF-IDF (2–5 grams)
username ───────────> character TF-IDF (2–5 grams)
biography ──────────> word TF-IDF (1–2 grams)

all features ───────> LinearSVC
```


## 7. Model training


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train);


## 8. Validation


In [ ]:
y_pred = model.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average="macro")
weighted_f1 = f1_score(y_val, y_pred, average="weighted")
accuracy = accuracy_score(y_val, y_pred)

print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"Accuracy:    {accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_val, y_pred, zero_division=0))


### Original validation result

In the original notebook, using the same 80/20 stratified split (`random_state=42`), the model achieved:

- **Macro F1: 0.9469**
- **Accuracy: ≈ 0.95**

Because the validation split contained equal support for the two classes, the weighted F1 is effectively the same as the macro F1 for that run.

Rerun the cleaned notebook to reproduce the metrics in your environment.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred,
)

plt.title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Final training and test inference

After validating the pipeline, retrain it on the full training set and generate test predictions.


In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            random_state=42,
        ),
    ),
])

final_model.fit(X, y);

test_pred = final_model.predict(test)

submission = pd.DataFrame({
    "gender": test_pred
})

submission.head()


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False,
)

print("Saved:", OUTPUT_DIR / "submission.csv")


## 10. Takeaways

This project demonstrates:

- heterogeneous feature preprocessing with `ColumnTransformer`
- robust missing-value handling inside the pipeline
- log transformation of skewed numerical features
- character-level TF-IDF for names and usernames
- word-level TF-IDF for free-text biography
- sparse text features combined with structured metadata
- linear SVM classification for high-dimensional sparse data
- reproducible train/validation evaluation

A useful next experiment would be a **small, time-bounded search over `LinearSVC(C)`** rather than an expensive search over all TF-IDF settings.
